# EXP006 — Path-Finding Algorithm Comparison

**Question:** Which path-finding algorithm surfaces the highest-quality introduction paths in the NetWeave graph?
**Hypothesis:** Dijkstra with inverted composite weights globally optimizes across the path, outperforming top_k_neighbors on multi-hop targets.
**Success criterion:** Winning algorithm achieves mean path composite ≥ 0.65 across 10 target nodes with path length ≤ 2 hops.
**Production impact:** `netweave/src/query.py` → path-finding function used in the goal-driven query engine.
**Author:** Chuck Wiley  **Date:** 2026-06-23

**Setup:** Set `CHUCK_NODE_ID` environment variable before running: `export CHUCK_NODE_ID="your_node_id"`

In [ ]:
import sys
sys.path.insert(0, ".")

import os
import mlflow
import mlflow.tracking
import networkx as nx
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from lib.niw_graph import load_graph
from lib.niw_mlflow import start_run, log_result, compare_runs

mlflow.set_tracking_uri("sqlite:///mlflow.db")

EXPERIMENT_ID = "EXP006"
EXPERIMENT_NAME = "Pathfinding Algorithms"
mlflow.set_experiment(EXPERIMENT_NAME)

CHUCK_NODE_ID = os.environ.get("CHUCK_NODE_ID")
if not CHUCK_NODE_ID:
    raise EnvironmentError("Set CHUCK_NODE_ID environment variable before running this notebook.")

In [ ]:
# Always load from shared/ — never modify source data
G = load_graph(
    nodes_path="shared/nodes.parquet",
    edges_path="shared/edges.parquet"
)
print(f"Graph loaded: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

if CHUCK_NODE_ID not in G.nodes:
    raise ValueError(f"CHUCK_NODE_ID '{CHUCK_NODE_ID}' not found in graph.")

# Build inverted-weight graph (higher composite = shorter path distance)
G_inv = G.copy()
for u, v, data in G_inv.edges(data=True):
    data["inv_weight"] = 1.0 - data.get("composite", 0.5)

print(f"Source node: {CHUCK_NODE_ID}")

In [ ]:
source = CHUCK_NODE_ID

def dijkstra_path(G_inv, target):
    try:
        return nx.dijkstra_path(G_inv, source, target, weight="inv_weight")
    except nx.NetworkXNoPath:
        return None

def astar_path(G_inv, target):
    def heuristic(u, v):
        return 1 - G.nodes[v].get("domain_score", 0.5)
    try:
        return nx.astar_path(G_inv, source, target, heuristic=heuristic, weight="inv_weight")
    except nx.NetworkXNoPath:
        return None

def top_k_neighbors(G, target, k=3):
    predecessors = list(G.predecessors(target))
    if not predecessors:
        return None
    best = sorted(
        predecessors,
        key=lambda n: G[n][target].get("tie_strength", 0),
        reverse=True
    )[:k]
    return [source, best[0], target] if best else None

# Manually select 10 hard-to-reach target nodes
# Replace with actual node IDs from your graph
TARGET_NODES = []  # e.g. ["n042", "n107", ...] — fill in before running
if not TARGET_NODES:
    raise ValueError("Fill in TARGET_NODES with 10 node IDs before running.")

variants = {
    "dijkstra":        lambda t: dijkstra_path(G_inv, t),
    "astar":           lambda t: astar_path(G_inv, t),
    "top_k_neighbors": lambda t: top_k_neighbors(G, t, k=3),
}

In [ ]:
def path_composite(G, path):
    """Product of composite edge weights along a path."""
    if path is None or len(path) < 2:
        return 0.0
    score = 1.0
    for i in range(len(path) - 1):
        score *= G[path[i]][path[i+1]].get("composite", 0.0)
    return score

results = []
for variant_name, path_fn in variants.items():
    with mlflow.start_run(run_name=f"{EXPERIMENT_ID}_{variant_name}"):
        mlflow.log_params({"algorithm": variant_name})

        path_scores, path_lengths, found = [], [], 0
        for target in TARGET_NODES:
            path = path_fn(target)
            if path is not None:
                found += 1
                path_scores.append(path_composite(G, path))
                path_lengths.append(len(path) - 1)  # hops
            else:
                path_scores.append(0.0)
                path_lengths.append(999)

        mean_composite = float(np.mean(path_scores))
        mean_length = float(np.mean(path_lengths))
        coverage = found / len(TARGET_NODES)

        mlflow.log_metrics({
            "mean_path_composite": mean_composite,
            "mean_path_length": mean_length,
            "path_coverage": coverage,
        })
        results.append({
            "variant": variant_name,
            "mean_composite": mean_composite,
            "mean_length": mean_length,
            "coverage": coverage,
        })
        print(f"{variant_name}: composite={mean_composite:.4f}, length={mean_length:.1f}, coverage={coverage:.2f}")

results_df = pd.DataFrame(results).sort_values("mean_composite", ascending=False)
print(results_df)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].barh(results_df["variant"], results_df["mean_composite"])
axes[0].set_xlabel("Mean Path Composite")
axes[0].set_title(f"{EXPERIMENT_ID}: Path Quality")
axes[0].invert_yaxis()

axes[1].barh(results_df["variant"], results_df["mean_length"])
axes[1].set_xlabel("Mean Path Length (hops)")
axes[1].set_title(f"{EXPERIMENT_ID}: Path Length")
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig("experiments/EXP006_pathfinding_algos/results.png", dpi=150)
plt.show()

## Result and Decision

**Winner:** [fill in after running]
**Mean path composite:** [fill in]
**Mean path length:** [fill in] hops
**Decision:** [VALIDATED | INCONCLUSIVE | REJECTED]

**If VALIDATED — production change:**
File: `netweave/src/query.py`
Function: Path-finding function used in the goal-driven query engine.
Change: Use winning algorithm for intro path generation.
Notebook citation: `# Validated in EXP006 — see netweave-lab/experiments/EXP006_pathfinding_algos/`